# 2. Preprocessing

In [1]:
import pandas as pd
from chess_eval.constants import *
from chess_eval import DataManager

As said in the first notebook (1_eda.ipynb), the only feature that the dataset contains is the FEN, which doesn't even count because it cannot be directly fed into a model. Therefore, the bigger part of this project will be spent on developing and engineering different features that hopefully will encapsulate all the necessary information to determine the winning side. Such, will be done in the next notebook (3_features.ipynb).

Apart from the cleaning done in the previous notebook, there is no other preprocessing needed for the dataset. We can automatize the cleaning process through a function `clean` (implemented in chess_eval/managers.py) that does just that. We can make use of the Data Manager and impute the cleaner right after reading the rows, so that it is automatically done for all samples.

In [2]:
dm = DataManager(cleaner=None)
dm.df_all.dtypes

FEN           object
Evaluation    object
dtype: object

In [3]:
def clean(df: pd.DataFrame, mode: str = "remove", threshold: int = EVAL_THRESHOLD) -> pd.DataFrame:
    df = df[~df[EVAL].astype(str).str.contains("#")].copy()
    df[EVAL] = pd.to_numeric(df[EVAL], errors="coerce").astype(int)
    if mode == "clip":
        df.loc[:, EVAL] = np.clip(df[EVAL], -threshold, threshold)
    elif mode == "remove":
        df = df[df[EVAL].abs() <= threshold]
    else:
        raise ValueError("mode must be 'clip' or 'remove'")
    return df

In [4]:
dm.df_all = clean(dm.df_all)
dm.df_all.dtypes

FEN           object
Evaluation     int64
dtype: object

So the evaluation column is transformed into numeric type just as we wanted. We also don't have any NaN's.

In [5]:
dm.df_all.isna().sum()

FEN           0
Evaluation    0
dtype: int64